# finance / economy / rates

The no risk rate to 2070 as six constants: nominal and real, low, base and high.

| Term | In This Notebook |
|---|---|
| **Intent** | Give one annual no risk (bank) rate and the matching real return of cash per scenario to 2070. Placeholder: low, base and high are the smallest, the mean and the largest of the 5, 10, 20 and 40 year means |
| **Project Flavour** | `static`, served by `mlserver` (`model.md` 3.2) |
| **Data Splits** | `training` the four window means of the bank rate and of the real cash return, synthetic until the `interest` table is read; `golden` the exact request that becomes `golden.json` |

## Setup

- **Stage:** none, preparation for the stages below.
- **Intent:** put `workbench` on the path, load `env`, and declare the model name, target, dataset names and signature names as constants (`model.md` 6.6).

In [ ]:
import json

from workbench.data import ensure_data, golden_data
from workbench.env import init_env
from workbench.project import provenance, record_project, scratch_project_dir

env = init_env()

WORKSPACE = "finance/economy/rates"
MODEL_NAME = "finance_economy_rates"
TARGETS = [{"kind": "served"}]
TRAINING = "economy_rates_training_2026-11-05_2026-11-05"
GOLDEN = "economy_rates_golden"
INPUT = "finance_economy_rates_in"
OUTPUTS = [
    "finance_economy_rates_nominal_low_pct",
    "finance_economy_rates_nominal_base_pct",
    "finance_economy_rates_nominal_high_pct",
    "finance_economy_rates_real_low_pct",
    "finance_economy_rates_real_base_pct",
    "finance_economy_rates_real_high_pct",
]

## Capture Data

- **Stage:** 1, capture data (`model.md` 7.1).
- **Intent:** write the `training` file only when it is absent (`model.md` 6.5). The source is synthetic until the real series is loaded, so no query runs.

In [ ]:
training_pl, training_path = ensure_data(WORKSPACE, "training", TRAINING, lambda: {"window_years": [5, 10, 20, 40], "bank_pct": [2.0, 2.6, 3.6, 5.0], "net_pct": [-0.9, -0.1, 0.8, 2.0]})
print("training", training_pl.height)

## Fit

- **Stage:** 3, fit.
- **Intent:** summarise the window means of `training` only (R6): the low is the smallest, the base the mean and the high the largest, until the real rate is modelled as mean reverting (see the plan).

In [ ]:
nominal, real = training_pl["bank_pct"].to_list(), training_pl["net_pct"].to_list()
CONSTANTS = dict(
    zip(
        OUTPUTS,
        [round(min(nominal), 2), round(sum(nominal) / len(nominal), 2), round(max(nominal), 2), round(min(real), 2), round(sum(real) / len(real), 2), round(max(real), 2)],
        strict=True,
    )
)
print("constants", CONSTANTS)
assert list(CONSTANTS.values()) == [2.0, 3.3, 5.0, -0.9, 0.45, 2.0], "the fit did not recover the window summary"

## Package

- **Stage:** 5, package.
- **Intent:** write the golden rows, build the signature, write a `static` project whose constants are the fitted values, verify it and replay the golden through the library loader (R1, R3).

In [ ]:
from workbench.model import build_signature, package_project, register_preview, signature_input, signature_output
from workbench.replay import require_replay

golden_pl, golden_path = golden_data(WORKSPACE, GOLDEN, lambda: {INPUT: [1.0, None], "expect": ["ok", "error"]})

signature = build_signature(
    MODEL_NAME,
    [signature_input(INPUT, "none", "Unused input.")],
    [
        signature_output("finance_economy_rates_nominal_low_pct", "percent", "Annual no risk rate, percent per year, low scenario.", datatype="FP64"),
        signature_output("finance_economy_rates_nominal_base_pct", "percent", "Annual no risk rate, percent per year, base scenario.", datatype="FP64"),
        signature_output("finance_economy_rates_nominal_high_pct", "percent", "Annual no risk rate, percent per year, high scenario.", datatype="FP64"),
        signature_output("finance_economy_rates_real_low_pct", "percent", "Annual real return of cash, percent per year, low scenario.", datatype="FP64"),
        signature_output("finance_economy_rates_real_base_pct", "percent", "Annual real return of cash, percent per year, base scenario.", datatype="FP64"),
        signature_output("finance_economy_rates_real_high_pct", "percent", "Annual real return of cash, percent per year, high scenario.", datatype="FP64"),
    ],
)
project_dir = scratch_project_dir(WORKSPACE)
manifest = package_project(
    project_dir,
    MODEL_NAME,
    "static",
    signature,
    golden_pl.iter_rows(named=True),
    provenance(
        "manual",
        "synthetic window means of the bank rate and real cash return, training split",
        [training_path, golden_path],
        "smallest, mean and largest of the window means",
    ),
    constants=CONSTANTS,
)
print(json.dumps(manifest, indent=2))

require_replay(project_dir)
metadata = record_project(WORKSPACE, project_dir, TARGETS)
print("replayed the golden sequences exactly through the shipped loader")

## Register

- **Stage:** 6, register.
- **Intent:** register the verified project with the alias `preview`. A person sets `release` after review and the parity gate (R5).

In [ ]:
version = register_preview(project_dir, MODEL_NAME, env["MLFLOW_TRACKING_URI"], TARGETS, metadata["snapshot_version"])
print("registered version", version)